# 🎓 Modelo de Machine Learning - Previsão de Evasão (EAD)
### a instituicao | Versão v3 (Foco em Alto Recall & Abordagem em Camadas)

Este notebook implementa o pipeline de Machine Learning para a modalidade **EAD** utilizando a base extraída do banco de dados institucional.

#### 🎯 Estrutura do Pipeline:
1. **Carregamento dos Dados:** Leitura da planilha extraída (`tabelas/extracao_ead_2025_74.xlsx`).
2. **Filtro de Elegibilidade:** Seleção exclusiva de alunos elegíveis (`ELEGIVEL == 1`).
3. **Engenharia de Features de Risco:** Indicadores de desempenho e comportamento no AVA.
4. **Split Estratificado:** 80% Treino e 20% Teste.
5. **Treinamento com Random Forest:** Ponderação de classes minoritárias (`class_weight`).
6. **Otimização do Limiar de Decisão (*Threshold Tuning*):** Foco em alto Recall para intervenção da retenção.
7. **Matrizes de Confusão Comparativas & Curva Precision-Recall.**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    recall_score,
    precision_score
)

# Configuracao visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Carregamento dos Dados
Lemos a planilha Excel extraída do banco para a modalidade EAD.

In [ ]:
file_path = '../../tabelas/extracao_ead_2025_74.xlsx'
if not os.path.exists(file_path):
    file_path = '../tabelas/extracao_ead_2025_74.xlsx'
if not os.path.exists(file_path):
    file_path = 'tabelas/extracao_ead_2025_74.xlsx'

print(f"Carregando dados de: {file_path}")
df = pd.read_excel(file_path)
print(f"Dimensões brutas: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(3)

## 2. Filtro Obrigatório de Elegibilidade (`ELEGIVEL == 1`)
Elimina quem já havia concluído, cancelado ou trancado antes do fechamento do período observado.

In [ ]:
df_model = df[df['ELEGIVEL'] == 1].copy()

total_elegiveis = len(df_model)
total_evasores = int((df_model['EVADIU'] == 1).sum())
taxa_evasao = df_model['EVADIU'].mean()

print(f"Total de alunos elegíveis: {total_elegiveis:,}")
print(f"Total de evasores reais:  {total_evasores:,}")
print(f"Taxa de evasão na base:   {taxa_evasao:.2%}")

## 3. Engenharia de Features de Risco Combinado
Criamos indicadores de risco cruzando desempenho acadêmico e comportamento no AVA.

In [ ]:
# 1. Normalizar estado civil
df_model['EST_CIVIL'] = df_model['EST_CIVIL'].astype(str).str.upper().replace({'NAN': 'NAO_INFORMADO', 'NONE': 'NAO_INFORMADO'})

# 2. Agrupar cauda longa de municípios (evita explosão de dummies com >400 cidades)
mun_counts = df_model['END_MUNICIPIO'].value_counts()
principais_mun = mun_counts[mun_counts >= 15].index.tolist()
df_model['END_MUNICIPIO_AGRUPADO'] = df_model['END_MUNICIPIO'].apply(lambda x: str(x) if x in principais_mun else 'OUTRO')
print(f"Municípios principais mantidos: {len(principais_mun)} (restante agrupado em 'OUTRO')")

# 3. Risco Acadêmico (sem diário de frequência, foco em rendimento real)
df_model['RISCO_NOTA_BAIXA'] = (df_model['NOTA_MEDIA'] < 5.0).astype(int)

# 4. Risco AVA: inatividade crítica
df_model['RISCO_SUMIU_AVA'] = (df_model['AVA_DIAS_DESDE_ULT_ACESSO'] > 30).astype(int)

# 5. Sinal Composto: Nota Baixa E Sumido do AVA
df_model['RISCO_COMBO_ACAD_AVA'] = df_model['RISCO_NOTA_BAIXA'] * df_model['RISCO_SUMIU_AVA']

# 6. Dinâmica temporal de login
df_model['RATIO_GAP_ATIVOS'] = df_model['AVA_MAIOR_GAP_DIAS'] / (df_model['AVA_DIAS_ATIVOS'] + 1)
df_model['SESSOES_POR_DIA_ATIVO'] = df_model['AVA_QTD_SESSOES'] / (df_model['AVA_DIAS_ATIVOS'] + 1)
df_model['DESACELEROU_AVA'] = (df_model['AVA_TEND_SESSOES'] < 0.5).astype(int)

print("✅ Engenharia de features refinada concluída com sucesso (sem ruídos)!")

## 4. Separação de $X$ (Features) e $y$ (Target)
Excluímos identificadores e variáveis que causam vazamento de dados.

In [ ]:
colunas_remover = [
    'ALUNO',                   # Identificador unico (overfitting)
    'CURSOU_ALVO',              # Insumo bruto do alvo (vazamento)
    'SIT_ALVO',                 # Derivado do alvo (vazamento)
    'ELEGIVEL',                 # Filtro de controle
    'SIT_FIM_OBS',              # Situacao de controle
    'AUDITORIA_STATUS',         # Auditoria de elegibilidade/desfecho
    'DT_INICIO_OBS',            # Metadado
    'DT_CORTE',                 # Metadado
    'ANO_OBS',                  # Metadado constante
    'PERIODO_OBS',              # Metadado constante
    'NOME_CURSO',               # Redundante com o codigo CURSO
    'EVADIU',                   # O rotulo alvo
    'NACIONALIDADE',            # 99.5% constante (sem sinal)
    'UNIDADE_FISICA',           # 99.7% constante (sem sinal)
    'END_MUNICIPIO',            # Substituido por END_MUNICIPIO_AGRUPADO
    'AVA_QTD_SESSOES',          # Redundante com SESSOES_POR_DIA_ATIVO
    'AVA_DIAS_ATIVOS',          # Redundante com AVA_TAXA_DIAS_ATIVOS
    'AVA_SESSOES_RECENTES',     # Redundante
    'AVA_DIAS_ATIVOS_RECENTES'  # Redundante
]

X = df_model.drop(columns=[c for c in colunas_remover if c in df_model.columns])
y = df_model['EVADIU'].astype(int)

print(f"Features preditivas limpas em X: {X.shape[1]}")
print("Features:", X.columns.tolist())

## 5. Encoding de Categóricas e Imputação

In [ ]:
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
print(f"Colunas categóricas ({len(cat_cols)}): {cat_cols}")

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Remover colunas 100% nulas caso existam (ex: faltas fisicas em modalidades a distancia)
cols_all_nan = [c for c in X_encoded.columns if X_encoded[c].isna().all()]
if cols_all_nan:
    print(f"Descartando {len(cols_all_nan)} colunas 100% nulas nesta modalidade: {cols_all_nan}")
    X_encoded = X_encoded.drop(columns=cols_all_nan)

print(f"Dimensões finais de X após encoding: {X_encoded.shape[1]} colunas")

## 6. Divisão Treino (80%) e Teste (20%) Estratificado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Treino: {X_train.shape[0]:,} amostras | Evasores: {y_train.sum():,} ({y_train.mean():.2%})")
print(f"Teste:  {X_test.shape[0]:,} amostras | Evasores: {y_test.sum():,} ({y_test.mean():.2%})")

## 7. Treinamento do Modelo Random Forest (Sensível a Custo)

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Calcular razao de desbalanceamento para os pesos
scale_pos = max(1.0, (len(y_train) - y_train.sum()) / max(1, y_train.sum()))

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight={0: 1.0, 1: float(scale_pos)},
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_imp, y_train)
print("✅ Modelo Random Forest treinado com sucesso!")

## 8. Otimização do Limiar de Decisão (*Threshold Tuning*)
Abaixo comparamos o desempenho do modelo variando o limiar de probabilidade:

In [ ]:
y_proba = rf_model.predict_proba(X_test_imp)[:, 1]

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]
tabela_thresh = []

for t in thresholds:
    y_p = (y_proba >= t).astype(int)
    cm = confusion_matrix(y_test, y_p)
    rec = recall_score(y_test, y_p, zero_division=0)
    prec = precision_score(y_test, y_p, zero_division=0)
    f1 = f1_score(y_test, y_p, zero_division=0)
    evasores_capturados = cm[1, 1] if cm.shape == (2,2) else 0
    evasores_perdidos = cm[1, 0] if cm.shape == (2,2) else y_test.sum()
    fp = cm[0, 1] if cm.shape == (2,2) else 0
    total_fila = y_p.sum()
    
    tabela_thresh.append({
        'Threshold': f"{t:.2f}",
        'Recall (Sensibilidade)': f"{rec:.2%}",
        'Precision': f"{prec:.2%}",
        'F1-Score': f"{f1:.2%}",
        'Evasores Capturados': f"{evasores_capturados} / {y_test.sum()}",
        'Evasores Perdidos (FN)': evasores_perdidos,
        'Falsos Alarmes (FP)': fp,
        'Fila Gerada': total_fila,
        '% da Base': f"{total_fila / len(y_test):.1%}"
    })

df_thresh = pd.DataFrame(tabela_thresh)
display(df_thresh)

## 9. Avaliação com Limiar Otimizado (Threshold = 0.35)

In [ ]:
OPTIMAL_THRESH = 0.35
y_pred_opt = (y_proba >= OPTIMAL_THRESH).astype(int)

print("="*65)
print(f"   RELATÓRIO DE CLASSIFICAÇÃO (THRESHOLD OTIMIZADO = {OPTIMAL_THRESH})")
print("="*65)
print(classification_report(y_test, y_pred_opt, target_names=['Permanência (0)', 'Evasão (1)'], zero_division=0))

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("="*65)
print(f"ROC-AUC Geral:                {roc_auc:.4f}")
print(f"PR-AUC (Average Precision):   {pr_auc:.4f}")
print("="*65)


## 10. Matrizes de Confusão Comparativas (0.50 vs 0.35)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz 1: Threshold Padrao (0.50)
y_pred_padrao = (y_proba >= 0.50).astype(int)
cm_padrao = confusion_matrix(y_test, y_pred_padrao)
sns.heatmap(cm_padrao, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Fica', 'Evadiu'], yticklabels=['Fica', 'Evadiu'])
axes[0].set_title(f'Limiar Padrão (0.50)\nRecall: {recall_score(y_test, y_pred_padrao, zero_division=0):.1%}', fontweight='bold')
axes[0].set_ylabel('Classe Real', fontweight='bold')
axes[0].set_xlabel('Classe Prevista', fontweight='bold')

# Matriz 2: Threshold Otimizado (0.35)
cm_opt = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False,
            xticklabels=['Fica', 'Evadiu'], yticklabels=['Fica', 'Evadiu'])
axes[1].set_title(f'Limiar Otimizado ({OPTIMAL_THRESH}) [RECOMENDADO]\nRecall: {recall_score(y_test, y_pred_opt, zero_division=0):.1%}', fontweight='bold')
axes[1].set_ylabel('Classe Real', fontweight='bold')
axes[1].set_xlabel('Classe Prevista', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Curva ROC e Curva Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color='#1F4E78', lw=2.5, label=f'Random Forest (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5)
axes[0].set_title('Curva ROC', fontweight='bold')
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall / TPR)')
axes[0].legend(loc='lower right')

# Curva Precision-Recall
prec_c, rec_c, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec_c, prec_c, color='#009a73', lw=2.5, label=f'Random Forest (PR-AUC = {pr_auc:.3f})')
axes[1].axhline(y_test.mean(), color='crimson', linestyle='--', lw=1.5, label=f'Baseline ({y_test.mean():.1%})')
axes[1].set_title('Curva Precision-Recall', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 12. Importância das Features (*Feature Importances*)

In [ ]:
feat_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Importancia': rf_model.feature_importances_
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=feat_df.head(20), 
    x='Importancia', 
    y='Feature', 
    hue='Feature',
    palette='viridis',
    legend=False
)
plt.title('Top 20 Features Mais Importantes - EAD', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Importância Relativa (Gini Importance)', fontsize=11, fontweight='bold')
plt.ylabel('Feature', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Salvamento do Artefato em Pickle (`modelo_evasao_ead.pkl`)
Salvamos o modelo treinado, o imputador de dados, as features e o limiar calibrado para inferência em lote.

In [ ]:
import joblib

pkl_path_ml = 'modelo_evasao_ead.pkl'
pkl_path_testes = os.path.join('..', '..', 'area-testes', 'ead', 'modelo_evasao_ead.pkl')

artifact = {
    'model': rf_model,
    'imputer': imputer,
    'feature_names': X_encoded.columns.tolist(),
    'threshold': 0.35,
    'principais_municipios': principais_mun,
    'colunas_remover': colunas_remover,
    'metadata': {
        'modalidade': 'EAD',
        'coorte': '2025/74',
        'n_features': X_encoded.shape[1],
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'recall_threshold_035': float(recall_score(y_test, y_pred_opt)),
        'precision_threshold_035': float(precision_score(y_test, y_pred_opt))
    }
}

joblib.dump(artifact, pkl_path_ml)
print(f"🎉 Modelo salvo em: {pkl_path_ml}")
if os.path.exists(os.path.dirname(pkl_path_testes)):
    joblib.dump(artifact, pkl_path_testes)
    print(f"🎉 Modelo copiado para: {pkl_path_testes}")


## 14. Matriz de Correlação das Features com a Evasão (Turno EAD)
Neste tópico final, analisamos a matriz de correlação linear de Pearson de todas as features numéricas e de telemetria do AVA AVA-web contra a variável dependente `EVADIU`.

In [ ]:
# 1. Matriz de correlacao das features numericas refinadas
num_cols = [c for c in X.select_dtypes(include=[np.number]).columns if c != 'CURSO']
df_corr = df_model[num_cols].copy()
df_corr['EVADIU'] = y

corr_ead = df_corr.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_ead, dtype=bool))
sns.heatmap(
    corr_ead, mask=mask, cmap='coolwarm', vmax=0.4, vmin=-0.4, center=0,
    square=True, linewidths=0.5, annot=True, fmt='.2f', cbar_kws={'shrink': 0.8}
)
plt.title(f"""Matriz de Correlação de Pearson — Graduação Turno EAD Refinado
(N = {len(df_model):,} alunos elegíveis)""", fontsize=12, fontweight="bold", pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

# 2. Ranking de correlacao com o alvo EVADIU
corr_all = X_encoded.corrwith(y).sort_values()
top_neg = corr_all.head(8)
top_pos = corr_all.tail(8)
ranking_alvo = pd.concat([top_neg, top_pos])

colors = ['#2B6CB0' if v < 0 else '#C53030' for v in ranking_alvo.values]

plt.figure(figsize=(12, 6))
bars = plt.barh(ranking_alvo.index, ranking_alvo.values, color=colors, edgecolor='black', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)

plt.title('Top Features: Correlação Linear com a Evasão no EAD (EVADIU)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Coeficiente de Correlação de Pearson (r)', fontsize=10, fontweight='bold')
plt.ylabel('Feature', fontsize=10, fontweight='bold')

for bar in bars:
    w = bar.get_width()
    offset = 0.005 if w >= 0 else -0.005
    ha = 'left' if w >= 0 else 'right'
    plt.text(w + offset, bar.get_y() + bar.get_height()/2, f"{w:+.2f}", 
             va='center', ha=ha, fontsize=9, fontweight='bold', color='#2D3748')

plt.xlim(min(ranking_alvo.values) - 0.05, max(ranking_alvo.values) + 0.05)
plt.tight_layout()
plt.show()

# 3. Diagnostico de Multicolinearidade (|r| >= 0.70)
corr_pairs = []
feat_cols = [c for c in df_corr.columns if c != 'EVADIU']
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        c1, c2 = feat_cols[i], feat_cols[j]
        val = corr_ead.loc[c1, c2]
        if abs(val) >= 0.70:
            corr_pairs.append({
                'Feature 1': c1, 
                'Feature 2': c2, 
                'Correlação': round(val, 4), 
                'Magnitude': abs(val)
            })

df_colin = pd.DataFrame(corr_pairs).sort_values('Magnitude', ascending=False).drop(columns=['Magnitude']) if corr_pairs else pd.DataFrame(columns=['Feature 1', 'Feature 2', 'Correlação'])
print('=' * 65)
print('📌 DIAGNÓSTICO DE MULTICOLINEARIDADE ALTA NO EAD (|r| >= 0.70)')
print('=' * 65)
print(df_colin.to_string(index=False) if not df_colin.empty else 'Nenhum par com multicolinearidade severa!')